# A Recurrent Network for Sentiment Classification

Open in [Google Colab](https://colab.research.google.com/github/febse/ta2025/blob/main/03-13-RNN-Sentiment-Classification.ipynb)

In the [previous chapter](03-12-RNN-Next-Word-Prediction.ipynb) a recurrent network read a sequence of words and predicted the next word. Here we use the same kind of layer to **classify a whole text**: the network reads the words one by one, and the final hidden state $h_T$ (see the recurrent cell $h_t = \tanh(x_t W_x + h_{t-1} W_h + b_h)$ in the previous chapter) serves as a summary of the review. The notation is the one of the previous chapters.

The numbers quoted in the comments below were obtained by running this code once with the seeds given in the code; they may differ slightly on another machine.

In [ ]:
import re
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import nltk
import torch
import torch.nn as nn

## Many-to-One: One Label for a Whole Text

In the previous chapter the network produced a word at every step. Here the output layer is applied only once, to the final hidden state $h_T$. It is the same logistic-regression output layer as in the [binary classification chapter](03-06-ANN-Logistic-Regression.ipynb),

:::{.ex-class}
$$
P(\text{positive} \mid w_1, \ldots, w_T) = \sigma\left( h_T W_o + b_o \right),
$$
:::

trained with the binary cross-entropy. In the [bag-of-words chapter](02-01-BOW-Models.ipynb) we classified the same product reviews (reviews of unlocked mobile phones) with a logistic regression on word counts. A bag of words ignores the order of the words, so "not good" and "good" differ only by one extra word "not". A recurrent network reads the words in order and can in principle learn that "not" changes the meaning of the next word.

### Data and Preprocessing

As in the earlier chapters we drop reviews with missing values and the neutral reviews with rating 3, and define the target as

$$
y = \begin{cases} 0 & \text{rating } 1 \text{ or } 2 \\ 1 & \text{rating } 4 \text{ or } 5 \end{cases}
$$

The text is preprocessed as in the [preprocessing chapter](02-02-Preprocessing.ipynb):

1. lower-case and split into words (letters only);
2. remove **stop words** (very frequent words like "the", "is", "of" that carry little meaning), **except negations**: the stop list of `nltk` contains "no", "nor", "not" and the contractions ("don't", "isn't", ...). Removing "not" would turn "not good" into "good", the opposite sentiment. We therefore expand "n't" to "not" first (don't $\to$ do not, can't $\to$ can not, won't $\to$ will not) and keep "no", "nor" and "not";
3. **stem** the remaining words with the Porter stemmer (batteries, battery $\to$ batteri);
4. keep the 30,000 most common stems; all other stems are replaced by `<unk>`.

Reviews are of different length. For a batch we need sequences of equal length, so each review is cut after 100 words and shorter reviews are filled up with a special `<pad>` token (index 0).

In [ ]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split

nltk.download("stopwords", quiet=True)

reviews = pd.read_csv("https://github.com/febse/data/raw/refs/heads/main/ta/reviews.csv")
reviews = reviews.dropna(subset=["Reviews", "Rating"])
reviews = reviews[reviews["Rating"] != 3]

review_texts = reviews["Reviews"].astype(str).tolist()
sentiment = (reviews["Rating"] > 3).astype(int).to_numpy()

print("reviews:", len(sentiment), " share of positive reviews:", sentiment.mean().round(3))
print(reviews["Rating"].value_counts().sort_index().to_dict())

In [ ]:
NEGATIONS = {"no", "nor", "not"}          # the negations in the stop list of nltk
STOP_WORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()
stem_cache = {}

def stem(word):
    if word not in stem_cache:
        stem_cache[word] = stemmer.stem(word)
    return stem_cache[word]

def preprocess(text, keep_negations=True):
    text = text.lower().replace("won't", "will not").replace("can't", "can not").replace("n't", " not")
    words = re.findall(r"[a-z]+", text)
    kept = [w for w in words if w not in STOP_WORDS or (keep_negations and w in NEGATIONS)]
    return [stem(w) for w in kept]

print(review_texts[1][:150])
print(preprocess(review_texts[1])[:25])
print(preprocess("The battery isn't good and I can't recommend it, no value for money"))

The next function turns the stemmed reviews into the arrays that the network needs: a matrix of word indices (padded with 0) and the true length of every review. The reviews are split randomly into training (70%), validation (15%) and test (15%) sets; the vocabulary is built from the training reviews only.

In [ ]:
MAX_WORDS = 100
MAX_VOCAB = 30000

def build_dataset(keep_negations=True, seed=0):
    docs = [preprocess(text, keep_negations) for text in review_texts]
    idx_tr, idx_rest = train_test_split(np.arange(len(docs)), test_size=0.3, random_state=seed, stratify=sentiment)
    idx_va, idx_te = train_test_split(idx_rest, test_size=0.5, random_state=seed, stratify=sentiment[idx_rest])

    stem_counts = Counter(w for i in idx_tr for w in docs[i])
    stems = ["<pad>", "<unk>"] + [w for w, _ in stem_counts.most_common(MAX_VOCAB)]
    stem_to_id = {w: i for i, w in enumerate(stems)}

    matrix = np.zeros((len(docs), MAX_WORDS), dtype=np.int64)
    lengths = np.zeros(len(docs), dtype=np.int64)
    for i, doc in enumerate(docs):
        encoded = [stem_to_id.get(w, 1) for w in doc][:MAX_WORDS] or [1]    # an empty review becomes one <unk>
        matrix[i, :len(encoded)] = encoded
        lengths[i] = len(encoded)
    return {"docs": docs, "stems": stems, "stem_to_id": stem_to_id, "X": torch.tensor(matrix),
            "lengths": torch.tensor(lengths), "y": torch.tensor(sentiment, dtype=torch.float32),
            "train": idx_tr, "val": idx_va, "test": idx_te, "n_distinct": len(stem_counts)}

review_data = build_dataset()
doc_lengths = np.array([len(d) for d in review_data["docs"]])

print("distinct stems in the training reviews:", review_data["n_distinct"], "-> vocabulary size (with <pad>, <unk>):", len(review_data["stems"]))
print("most common stems:", review_data["stems"][2:17])
print(f"median length: {np.median(doc_lengths):.0f} words, reviews cut at {MAX_WORDS} words: {(doc_lengths > MAX_WORDS).mean():.1%}")
print("train / val / test:", len(review_data["train"]), len(review_data["val"]), len(review_data["test"]))

There are 45,831 reviews and 74.5% of them are positive. After removing stop words the median review has only 9 words and 3.3% of the reviews are longer than 100 words and get cut. The training reviews contain only 15,068 distinct stems, so the limit of 30,000 words is not binding here: every stem gets its own embedding, and only stems that occur exclusively in the validation or test reviews become `<unk>`. (For a larger corpus the limit would matter, because a rare word that occurs once or twice cannot be learned anyway.) Note that "not" and "no" are among the ten most frequent stems, which shows how common negations are in reviews.

### The Network

The class below is the many-to-one network: embedding, recurrent layer, and a linear output with one logit. Two details are new:

- **Padding.** The padded positions must not influence the result. With `pack_padded_sequence` the recurrent layer stops at the true length of each review, so the final state $h_T$ belongs to the last real word and not to the padding.

- **Gradient clipping** is explained below.

### Gradient Clipping

**The problem.** Backpropagation through time sends the gradient back through all $T$ time steps, and at every step it is multiplied by the same matrix $W_h$ (and by the derivative of the activation function). For a review of $T$ words the gradient therefore behaves roughly like $W_h^T$. If the largest singular value of $W_h$ is larger than 1, the gradient grows exponentially with the length of the review (**exploding gradients**): a single mini-batch can produce an enormous gradient, the update $\theta \leftarrow \theta - \eta g$ throws the weights far away, and the loss jumps up or becomes `nan`. (If the largest singular value is smaller than 1, the gradient shrinks instead, the **vanishing gradient** problem: the network cannot learn from the early words. The gates of the GRU and the LSTM address that case; clipping does not.)

**The solution.** Let $g$ be the gradient of the loss with respect to all parameters, stacked into one vector, and $c$ a threshold (we use $c = 5$). Before the update we rescale the gradient if it is too long:

:::{.ex-class}
$$
g \leftarrow \begin{cases} g & \text{if } \lVert g \rVert_2 \le c \\[4pt] c \, \dfrac{g}{\lVert g \rVert_2} & \text{if } \lVert g \rVert_2 > c \end{cases}
$$
:::

A normal gradient is not changed. A gradient that is too large keeps its **direction**, so the step still goes downhill, but its **length** is cut to $c$, so one bad batch cannot destroy the weights. In PyTorch this is one line between the backward pass and the optimizer step: `nn.utils.clip_grad_norm_(model.parameters(), clip)`. The value 5 is a common default, not a tuned one.

In [ ]:
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, kind="rnn", embedding_size=64, hidden_size=64):
        super().__init__()
        self.kind = kind
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=0)
        layer = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.recurrent = layer(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, word_ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(word_ids), lengths, batch_first=True, enforce_sorted=False)
        _, last_state = self.recurrent(packed)
        if self.kind == "lstm":
            last_state = last_state[0]                      # the LSTM also returns its cell state
        return self.output(last_state[-1]).squeeze(1)       # one logit per review

    def logits_after_each_word(self, word_ids):
        # for ONE review (1, T): the logit that we would get if the review ended after word t
        states, _ = self.recurrent(self.embedding(word_ids))
        return self.output(states[0]).squeeze(1)

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def split_tensors(data, name):
    rows = torch.tensor(data[name])
    return data["X"][rows], data["lengths"][rows], data["y"][rows]

def predict_logits(model, X, lengths, batch_size=512):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[s:s + batch_size], lengths[s:s + batch_size]) for s in range(0, len(X), batch_size)])

def fit_sentiment(model, data, n_epochs=8, learning_rate=2e-3, batch_size=64, clip=5.0, seed=0):
    X_tr, len_tr, y_tr = split_tensors(data, "train")
    X_va, len_va, y_va = split_tensors(data, "val")
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"val": []}
    best_loss, best_state = float("inf"), None
    for epoch in range(n_epochs):
        model.train()
        order = torch.randperm(len(X_tr))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            bce_loss(model(X_tr[batch], len_tr[batch]), y_tr[batch]).backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        history["val"].append(bce_loss(predict_logits(model, X_va, len_va), y_va).item())
        if history["val"][-1] < best_loss:
            best_loss, best_state, history["best_epoch"] = history["val"][-1], {k: v.clone() for k, v in model.state_dict().items()}, epoch
    model.load_state_dict(best_state)
    return model, history

def test_scores(model, data):
    X_te, len_te, y_te = split_tensors(data, "test")
    probs = torch.sigmoid(predict_logits(model, X_te, len_te)).numpy()
    return probs, y_te.numpy()

### Training and Comparison

We train the simple recurrent network and the GRU and compare them with the bag-of-words approach of the earlier chapter: a logistic regression on TF-IDF weights of the same stemmed words (the regularization strength is chosen on the validation set). Because about three quarters of the reviews are positive, the accuracy of always predicting "positive" is already high; we also report the area under the ROC curve (AUC), which does not depend on the class proportions.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

sentiment_models = {}
for kind in ["rnn", "gru"]:
    torch.manual_seed(0)
    sentiment_models[kind] = fit_sentiment(SentimentNet(len(review_data["stems"]), kind), review_data)

# bag-of-words baseline on the same stems
tfidf = TfidfVectorizer(analyzer=lambda tokens: tokens, min_df=2, max_features=MAX_VOCAB)
docs_train = [review_data["docs"][i] for i in review_data["train"]]
docs_val = [review_data["docs"][i] for i in review_data["val"]]
docs_test = [review_data["docs"][i] for i in review_data["test"]]
B_train, B_val, B_test = tfidf.fit_transform(docs_train), tfidf.transform(docs_val), tfidf.transform(docs_test)
y_train_np, y_val_np, y_test_np = (sentiment[review_data[s]] for s in ["train", "val", "test"])

best_c = max([0.3, 1, 3, 10, 30], key=lambda c: LogisticRegression(C=c, max_iter=2000).fit(B_train, y_train_np).score(B_val, y_val_np))
tfidf_logreg = LogisticRegression(C=best_c, max_iter=2000).fit(B_train, y_train_np)

rows = [("majority class", max(y_test_np.mean(), 1 - y_test_np.mean()), 0.5, "-")]
rows.append(("TF-IDF + logistic regression", tfidf_logreg.score(B_test, y_test_np),
             roc_auc_score(y_test_np, tfidf_logreg.predict_proba(B_test)[:, 1]), "-"))
for kind, (model, history) in sentiment_models.items():
    probs, truth = test_scores(model, review_data)
    rows.append((f"{kind.upper()} (hidden size 64)", ((probs > 0.5) == truth).mean(), roc_auc_score(truth, probs), history["best_epoch"] + 1))

print(f"{'model':<30} {'test accuracy':>14} {'test AUC':>9} {'best epoch':>11}")
for name, acc, auc, epoch in rows:
    print(f"{name:<30} {acc:>14.3f} {auc:>9.3f} {str(epoch):>11}")
print("selected C of the logistic regression:", best_c)

The majority-class baseline reaches 74.5% accuracy, so all models are clearly useful. The GRU (93.8% accuracy, AUC 0.976) is on par with the bag-of-words logistic regression (93.6%, AUC 0.978), and the simple recurrent network is somewhat worse (92.1%, AUC 0.961). So for this task a recurrent network gives **no real advantage** over the bag of words: most of the sentiment is carried by individual words ("great", "love", "broke"), and the typical review is short. The difference between the simple RNN and the GRU is plausible (the gates help with longer reviews and with gradients), but we have trained each network only once and the differences of a few tenths of a percentage point are within the noise. The GRU reached its best validation loss after only 2 epochs and then overfitted; the simple RNN was still improving slowly at epoch 8.

### Reading a Review Word by Word

Because the hidden state is updated after every word, we can apply the output layer after **every** word and watch how the predicted probability of a positive review evolves. We use two new reviews: one with a clear turn in the middle and one with a negation.

In [ ]:
def word_by_word(model, text):
    words = preprocess(text)
    ids_ = torch.tensor([[review_data["stem_to_id"].get(w, 1) for w in words]])
    model.eval()
    with torch.no_grad():
        return words, torch.sigmoid(model.logits_after_each_word(ids_)).numpy()

gru = sentiment_models["gru"][0]
examples = ["The screen is great and the camera is nice but the battery is terrible and it does not hold a charge",
            "Not bad at all, works great and I would buy it again"]

fig, axes = plt.subplots(len(examples), 1, figsize=(10, 6))
for ax, text in zip(axes, examples):
    words, probs = word_by_word(gru, text)
    ax.plot(probs, marker="o")
    ax.axhline(0.5, color="gray", linestyle=":")
    ax.set(ylim=(0, 1), ylabel="P(positive)", xticks=range(len(words)))
    ax.set_xticklabels(words, rotation=60, fontsize=8)
    ax.set_title(text, fontsize=9)
plt.tight_layout()
plt.show()

The curves show how the network "reads". In the first review the probability jumps up with "great", stays high through "camera nice battery" and starts to fall at "terribl" and "not", ending slightly below 0.5 at the end of the review (a mixed review that the network finds on balance slightly negative). In the second review the network is **wrong at the beginning and recovers at the end**: after "not bad" the probability is very low, i.e. the network treats "not bad" as negative instead of positive, and only the later words "great ... would buy" bring the prediction up (0.69 at the end). The network has learned that "not" is associated with negative reviews (it is mostly followed by "work", "good", "recommend"), but it has not learned the rule that a negation reverses the meaning of the *next* word; we could not expect that from a few thousand short reviews.

### Does Keeping the Negations Matter?

We repeat the whole experiment with the negations **removed** together with the other stop words and compare the test accuracy of the GRU. We also check simple sentences with and without negation (preprocessed in the way each model was trained).

In [ ]:
# no-ex-class
data_without_negations = build_dataset(keep_negations=False)
torch.manual_seed(0)
gru_without, _ = fit_sentiment(SentimentNet(len(data_without_negations["stems"]), "gru"), data_without_negations)

probs_without, truth_without = test_scores(gru_without, data_without_negations)
probs_with, truth_with = test_scores(gru, review_data)
print(f"GRU, negations kept   : accuracy {((probs_with > 0.5) == truth_with).mean():.3f}, AUC {roc_auc_score(truth_with, probs_with):.3f}")
print(f"GRU, negations removed: accuracy {((probs_without > 0.5) == truth_without).mean():.3f}, AUC {roc_auc_score(truth_without, probs_without):.3f}")

def positive_probability(model, data, text, keep_negations):
    words = [w for w in (preprocess(text, keep_negations))]
    ids_ = torch.tensor([[data["stem_to_id"].get(w, 1) for w in words]])
    return torch.sigmoid(model(ids_, torch.tensor([ids_.shape[1]]))).item()

print()
print(f"{'sentence':<42} {'negations kept':>15} {'negations removed':>18}")
for text in ["the phone is good", "the phone is not good", "works great", "does not work", "i like it", "i do not like it", "no problems at all"]:
    print(f"{text:<42} {positive_probability(gru, review_data, text, True):>15.2f} "
          f"{positive_probability(gru_without, data_without_negations, text, False):>18.2f}")

Keeping the negations helps: the test accuracy of the GRU is 93.8% with and 92.8% without them (AUC 0.976 against 0.968). This is a single run per setting, so the difference of one percentage point should be treated as suggestive, not as proven. The sentence examples show the mechanism very clearly. With the negations removed "the phone is not good" becomes "phone good" and gets a probability of 0.96 of being positive; "i do not like it" gets 0.89. With the negations kept, these sentences get 0.09 and 0.16. For "no problems at all" both models give about 0.83, because "problem" alone is negative and "no" is common in both positive and negative reviews.

### Where Does the Network Fail?

Finally we look at reviews that the GRU classifies with high confidence but wrongly.

In [ ]:
# no-ex-class
probs, truth = test_scores(gru, review_data)
test_rows = review_data["test"]
wrong = np.where((probs > 0.5) != truth)[0]
most_confident = wrong[np.argsort(-np.abs(probs[wrong] - 0.5))][:5]
for j in most_confident:
    print(f"rating {int(reviews['Rating'].iloc[test_rows[j]])}, P(positive) = {probs[j]:.2f}: {review_texts[test_rows[j]][:220]!r}\n")

Most of the confident mistakes are not mistakes of the network at all but **label noise**: "Its working just fine i rate it a 5 star" was given a rating of 1, "nice for its price" a rating of 2, "For the price it's a good buy" a rating of 2. The text is clearly positive and so is the network's prediction. The ratings come from the reviewers, who sometimes click the wrong number of stars or write about a different aspect than the rating (the "LTE works great on Metro but the phone has software issues" review mixes both). This sets a limit on the accuracy that any model can reach on this data, and a good reason not to compare models that differ by a few tenths of a percentage point.

## Limitations

- **The task is easy for a bag of words.** The GRU is on par with logistic regression on TF-IDF features, because the sentiment of short reviews is carried by individual words. Recurrent networks pay off when word order matters more, or with much more data.
- **Negation is only partly understood.** The network learns that "not" is associated with negative reviews, but it does not reliably reverse the meaning of the next word ("not bad").
- **Noisy labels.** The star ratings are noisy, which limits the achievable accuracy.
- **A single run.** All comparisons are based on one training run per model; differences of a few tenths of a percentage point are not meaningful.

## Summary

- For classification the final hidden state of the recurrent layer feeds a logistic-regression output layer trained with the binary cross-entropy.
- Variable-length texts need padding and packing, and recurrent networks need gradient clipping.
- Stop words, stemming and a limited vocabulary shrink the problem, but removing negations such as "not" changes the meaning of a sentence and hurts the classifier.
- A GRU matches the bag-of-words baseline on this data and a simple RNN is slightly behind.

## Exercises

1. Raise `MAX_WORDS` to 300. How many more reviews are no longer truncated, and does the accuracy change?
2. Replace the GRU by an LSTM (`kind="lstm"`) and compare.
3. Use a **bidirectional** recurrent layer (`bidirectional=True`; the final output layer then needs twice as many inputs). Does reading the review in both directions help with "not bad"?
4. Predict the rating on the scale 1 to 5 (without the neutral reviews removed) with a softmax output, see the [multi-class chapter](03-07-ANN-Multiclass-Classification.ipynb).
5. Add the negations "never", "nothing" and "nobody" to `NEGATIONS`. These words are not in the stop list of `nltk`, so they are already kept; why can it still make sense to look at them separately?